# 08 · Attack type for each flagged request

The detector says *attack* or *normal*. An investigation also needs the kind of attack: **sqli**, **xss**, **traversal**, **cmdi**, or **scan** (a probe for a hidden file). `ml/attack_type.py` does this in two parts:

1. **Rules:** for each family, count how many of its patterns (the same ones as in `features.py`) match. The family with the most matching patterns wins.
2. **A small model** for the cases the rules cannot decide (nothing matches, or two families tie). It is trained on the attack types of HttpParams (public data), never on our generator.

One domain rule: a tie between cmdi and traversal (a shell command that reads `/etc/passwd`) is cmdi.

**Data:** run3 is development data and was used to design the tie rules, so its numbers are optimistic. The HttpParams test split was never trained on. The honest test is run4 and the third site.

In [1]:
import sys
from pathlib import Path
from urllib.parse import quote
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from sklearn.utils.class_weight import compute_sample_weight
from ml.attack_type import TYPES, _pick, rule_scores
from ml.detector import to_frame
from ml.train_attack_type import build_model, load
from eval.match_labels import match

Xtr, ytr = load(ROOT / 'data' / 'raw' / 'httpparams_train.csv')
type_model = build_model().fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))

hp = pd.read_csv(ROOT / 'data' / 'raw' / 'httpparams_test.csv', keep_default_na=False)
hp = hp[hp['label'] == 'anom']
hp = pd.DataFrame({'method': 'GET', 'path': ['/?q=' + quote(v, safe='') for v in hp['payload']], 'body': '',
                   'attack_type': hp['attack_type'].replace({'path-traversal': 'traversal'}).values})
run3, _ = match(ROOT / 'data' / 'raw' / 'juiceshop_generated_run3.log', ROOT / 'data' / 'labels' / 'requests_run3.csv')
run3_payload = run3[run3['attack_type'].isin(TYPES)]
sets = {'run3 (dev)': run3_payload, 'HttpParams test': hp}
{k: v['attack_type'].value_counts().to_dict() for k, v in sets.items()}

{'run3 (dev)': {'cmdi': 97, 'sqli': 76, 'xss': 59, 'traversal': 58},
 'HttpParams test': {'sqli': 3617, 'xss': 177, 'traversal': 97, 'cmdi': 30}}

## 1. Rules only, model only, and both together

Scored on true attacks only: the question here is the kind, not whether it is an attack.

In [2]:
def predict(d, how):
    X = to_frame(d['method'], d['path'], d['body']).drop(columns='text')
    proba = type_model.predict_proba(X)
    if how == 'model only':
        return [TYPES[i] for i in proba.argmax(1)]
    use = proba if how == 'rules + model' else [None] * len(d)
    return [_pick(rule_scores(m, p, b), pr) for m, p, b, pr in zip(d['method'], d['path'], d['body'], use)]

HOWS = ['rules only', 'model only', 'rules + model']
preds = {(s, h): predict(d, h) for s, d in sets.items() for h in HOWS}
pd.DataFrame({h: {s: (pd.Series(preds[(s, h)], index=d.index) == d['attack_type']).mean() for s, d in sets.items()}
              for h in HOWS}).round(3)

,rules only,model only,rules + model
run3 (dev),0.952,0.714,0.959
HttpParams test,0.923,0.995,0.995


- **The model alone** is very good on HttpParams (the data it learned from) but weak on run3: there, traversal sits in the URL path, which HttpParams never shows it.
- **The rules alone** are good on run3 but often tie on HttpParams SQLi that joins strings with `||` (it looks like a shell pipe).
- **Together** they keep the strength of both, so `attack_types()` uses rules + model.

## 2. Where the combined method goes wrong

In [3]:
pd.concat({s: pd.crosstab(d['attack_type'], pd.Series(preds[(s, 'rules + model')], index=d.index), rownames=['true'], colnames=['predicted']) for s, d in sets.items()})

predicted                  cmdi  sqli  traversal  xss
                true                                 
run3 (dev)      cmdi         93     4          0    0
                sqli          2    70          0    4
                traversal     0     0         58    0
                xss           0     2          0   57
HttpParams test cmdi         27     1          1    1
                sqli          3  3610          4    0
                traversal     6     0         91    0
                xss           1     2          1  173

## 3. Limitations

- One request can carry two kinds at once (a command that also reads a file). We report one: the strongest.
- A flagged request where no pattern matches gets the model's best guess. It can be wrong, and the investigation should show it as the model's guess.
- Brute force, ID enumeration and scanning by 404s come from the session layer (notebook 06), not from this module.